# CIFAKE：AI 生成图像检测

## 学习目标
1. SVM：实现 hinge loss 与线性 SVM 的小批量随机次梯度训练，用随机傅里叶特征近似 RBF 核
2. K-Means：实现 k-means++ 初始化与 Lloyd 迭代，用聚类做异常检测，实现 AUROC 指标
3. FFN：实现 MLP 及训练、验证、模型选择的完整流程
4. CNN：实现 CNN 与数据增强，比较优化器与正则化的效果

## CIFAKE

CIFAKE（Bird & Lotfi, 2023）包含 60,000 张 CIFAR-10 真实照片（`REAL`）和 60,000 张 Stable Diffusion v1.4 生成的同类别图像（`FAKE`），32×32 RGB。训练集 100,000 张，测试集 20,000 张，两类各半。任务是判断一张图片是照片还是生成图。标签：`0` 为 REAL，`1` 为 FAKE。

数据从 Kaggle 下载：[`birdy654/cifake-real-and-ai-generated-synthetic-images`](https://www.kaggle.com/datasets/birdy654/cifake-real-and-ai-generated-synthetic-images)。解压到本作业同级的 `data/cifake/`：

```
data/cifake/
├── train/REAL/*.jpg   (50,000)
├── train/FAKE/*.jpg   (50,000)
├── test/REAL/*.jpg    (10,000)
└── test/FAKE/*.jpg    (10,000)
```

## Guidelines
- 在 `# ========== 开始编写代码 ==========` 和 `# ========== 结束编写代码 ==========` 之间完成实验代码。尽量都用 PyTorch 原生实现。
- 完成所有实验在 H200 上需 10 分钟，在普通显卡上预计也在半小时内。

## Submission
- 运行完整、保留全部输出的本 notebook，与所有生成结果（图片，CSV）打包的压缩文件。需要填写完毕 Part 5 的所有问题。

In [ ]:
import math, time, csv, copy, random
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import matplotlib.pyplot as plt
%matplotlib inline
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision.io import read_image, ImageReadMode

print(torch.__version__, torchvision.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")

SEED = 0
torch.manual_seed(SEED)
random.seed(SEED)

DATA_ROOT = Path("data/cifake")
OUT_DIR = Path("output")
OUT_DIR.mkdir(exist_ok=True)

N_TRAIN = None       # 神经网络使用的训练样本数，None 表示全部
N_CLASSIC = 20_000   # SVM / K-Means 使用的训练子集大小
VAL = 5_000          # 从训练集划出的验证集大小
EPOCHS = 10
BATCH = 256

results = []

def record(part, name, metric, value, seconds):
    results.append(dict(part=part, method=name, metric=metric, value=round(float(value), 4), seconds=round(seconds, 1)))
    print(f"[{part}] {name}: {metric} = {value:.4f}  ({seconds:.0f}s)")

def savefig(name):
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"{name}.png", dpi=120, bbox_inches="tight")
    plt.show()

## Part 0: 数据准备

读取 CIFAKE 中的 120,000 张图片并缓存到 `data/cifake.pt` 中。图片形状为 `[N, 3, 32, 32]`。

In [ ]:
def load_split(split):
    xs, ys = [], []
    for label, name in ((0, "REAL"), (1, "FAKE")):
        files = sorted((DATA_ROOT / split / name).glob("*.jpg"))
        assert files, f"{DATA_ROOT / split / name} 下没有图片，请检查数据路径"
        with ThreadPoolExecutor(16) as pool:
            imgs = list(pool.map(lambda f: read_image(str(f), ImageReadMode.RGB), files))
        xs.append(torch.stack(imgs))
        ys.append(torch.full((len(imgs),), label))
    return torch.cat(xs), torch.cat(ys)

cache = DATA_ROOT.with_suffix(".pt")
if cache.exists():
    X_tr, y_tr, X_te, y_te = torch.load(cache)
else:
    t = time.time()
    X_tr, y_tr = load_split("train")
    X_te, y_te = load_split("test")
    torch.save((X_tr, y_tr, X_te, y_te), cache)
    print(f"读取完成，用时 {time.time() - t:.0f}s")

print("train:", tuple(X_tr.shape), X_tr.dtype, " 标签计数:", torch.bincount(y_tr).tolist())
print("test: ", tuple(X_te.shape), X_te.dtype, " 标签计数:", torch.bincount(y_te).tolist())

In [ ]:
CLASS_NAMES = ("REAL", "FAKE")

fig, axes = plt.subplots(2, 10, figsize=(14, 3.2))
for row, label in enumerate((0, 1)):
    idx = torch.nonzero(y_tr == label).squeeze(1)
    idx = idx[torch.randperm(len(idx))[:10]]
    for ax, i in zip(axes[row], idx):
        ax.imshow(X_tr[i].permute(1, 2, 0))
        ax.set_axis_off()
        ax.set_title(CLASS_NAMES[label], fontsize=8)
savefig("samples")

### 预处理与数据划分

In [ ]:
X_tr, y_tr, X_te, y_te = X_tr.to(device), y_tr.to(device), X_te.to(device), y_te.to(device)

MEAN = (X_tr.float() / 255).mean((0, 2, 3)).view(1, 3, 1, 1)
STD = (X_tr.float() / 255).std((0, 2, 3)).view(1, 3, 1, 1)

def normalize(X):
    return (X.float() / 255 - MEAN) / STD

T_tr, T_te = normalize(X_tr), normalize(X_te)
x_tr, x_te = T_tr.flatten(1), T_te.flatten(1)

perm = torch.randperm(len(T_tr), device=device)
val_idx, fit_idx = perm[:VAL], perm[VAL:]
if N_TRAIN is not None:
    fit_idx = fit_idx[:N_TRAIN]
sub = fit_idx[:N_CLASSIC]

print(f"训练 {len(fit_idx)} / 验证 {len(val_idx)} / 测试 {len(T_te)}；经典方法子集 {len(sub)}")
print("通道均值:", MEAN.flatten().tolist(), " 通道标准差:", STD.flatten().tolist())

### 辅助函数：PCA

Part 1 和 Part 2 都会先用 PCA 降维。`pca_fit` 返回数据均值和前 `k` 个主方向（也就是协方差矩阵的特征向量，按特征值从大到小排列），`pca_transform` 把数据投影到这些方向上，`pca_inverse` 则从投影还原回原来的空间。

In [ ]:
def pca_fit(X, k):
    mean = X.mean(0)
    cov = (X - mean).T @ (X - mean) / (len(X) - 1)
    evals, evecs = torch.linalg.eigh(cov)                 # 升序
    return mean, evecs[:, -k:].flip(1)                    # 取最大的 k 个并改为降序

def pca_transform(X, mean, V):
    return (X - mean) @ V

def pca_inverse(Z, mean, V):
    return Z @ V.T + mean

mean_sub, V_sub = pca_fit(x_tr[sub], 100)
z = pca_transform(x_te, mean_sub, V_sub)
print("PCA-100 测试集投影:", tuple(z.shape))
print("前 10 个主成分的方差:", [round(v, 2) for v in z.var(0)[:10].tolist()])

## Part 1: 支持向量机

### 任务1.1：hinge loss

软间隔线性 SVM 的原问题如下（$y_i\in\{-1,+1\}$）：

$$
\min_{w,b}\ \frac{\lambda}{2}\|w\|^2 + \frac{1}{N}\sum_{i=1}^{N}\max\bigl(0,\ 1 - y_i(w^\top x_i + b)\bigr)
$$

第二项就是 hinge loss。它在 $1 - y(w^\top x + b) = 0$ 处不可导，不过不需要特殊处理，前向直接用 `torch.clamp` 写出来，反向传播得到的就是一个次梯度，用它做小批量随机梯度下降即可（也就是 Pegasos 算法）。

作业里的标签是 `0/1`，用之前要先转成 `±1`，写成 `y_pm = 2 * y - 1`，这样 REAL 对应 −1，FAKE 对应 +1。

In [ ]:
# TODO: 任务1.1 - 实现合页损失
def hinge_loss(scores, y_pm):
    """
    合页损失（对 batch 取平均）

    Args:
        scores: torch.Tensor, shape [batch]，模型输出 w^T x + b
        y_pm:   torch.Tensor, shape [batch]，取值 -1 或 +1

    Returns:
        loss: 标量，mean(max(0, 1 - y * score))

    提示：
    1. 先计算 margin = y_pm * scores
    2. torch.clamp(1 - margin, min=0) 再取平均
    """
    # ========== 开始编写代码 ==========



    # ========== 结束编写代码 ==========
    pass

# 测试你的实现
def test_hinge_loss():
    print("测试 hinge_loss")
    print("=" * 50)
    scores = torch.tensor([2.0, 0.5, -1.0, -0.2])
    y_pm = torch.tensor([1.0, 1.0, -1.0, 1.0])
    loss = hinge_loss(scores, y_pm)
    print(f"你的输出: {loss.item():.4f}  期望: 0.4250")
    print(f"是否一致: {torch.allclose(loss, torch.tensor(0.425))}")
    print(f"正确分类且在间隔外的样本损失为 0: {hinge_loss(torch.tensor([3.0]), torch.tensor([1.0])).item() == 0.0}")
    print("=" * 50)

test_hinge_loss()

### 任务1.2：线性 SVM 的训练与预测

用一个 `nn.Linear(d, 1)` 来表示 $w$ 和 $b$，再用小批量随机次梯度下降最小化上面的目标函数。训练数据是 `N_CLASSIC` 张图片标准化之后的像素，每张展平成 3072 维。

In [ ]:
# TODO: 任务1.2 - 线性 SVM 的训练与预测
def train_linear_svm(X, y, lam=1e-4, epochs=20, lr=1e-3, batch_size=256, log=False):
    """
    小批量随机次梯度下降训练线性 SVM

    Args:
        X: torch.Tensor, shape [N, d]
        y: torch.Tensor, shape [N]，取值 0/1
        lam: L2 正则系数 λ

    Returns:
        model: nn.Linear(d, 1)

    提示：
    1. y_pm = 2 * y - 1
    2. 优化器使用 torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    3. 每个 epoch 用 torch.randperm 打乱样本，按 batch_size 切片
    4. 目标函数 = hinge_loss(model(xb).squeeze(1), yb) + 0.5 * lam * (model.weight ** 2).sum()
    """
    model = nn.Linear(X.shape[1], 1).to(X.device)
    # ========== 开始编写代码 ==========



    # ========== 结束编写代码 ==========
    return model

@torch.no_grad()
def svm_predict(model, X):
    """返回 0/1 预测标签，shape [N]"""
    # ========== 开始编写代码 ==========



    # ========== 结束编写代码 ==========
    pass

@torch.no_grad()
def svm_accuracy(model, X, y):
    return (svm_predict(model, X) == y).float().mean().item()

# 一个小规模的线性可分测试
def test_linear_svm():
    print("测试 train_linear_svm")
    print("=" * 50)
    g = torch.Generator().manual_seed(0)
    Xs = torch.randn(400, 2, generator=g)
    ys = (Xs[:, 0] + Xs[:, 1] > 0).long()
    model = train_linear_svm(Xs.to(device), ys.to(device), lam=1e-3, epochs=30, lr=0.05)
    acc = svm_accuracy(model, Xs.to(device), ys.to(device))
    print(f"线性可分数据上的训练准确率: {acc:.4f}（应 > 0.97）")
    print("=" * 50)

test_linear_svm()

In [ ]:
t = time.time()
svm_raw = train_linear_svm(x_tr[sub], y_tr[sub], log=True)
record("Part 1", "linear SVM (raw pixels)", "acc", svm_accuracy(svm_raw, x_te, y_te), time.time() - t)

### 任务1.3：随机傅里叶特征近似 RBF 核

核 SVM 需要计算 $N\times N$ 的核矩阵，十万张图片的话内存根本放不下。随机傅里叶特征（Random Fourier Features, RFF）的想法是给 RBF 核 $k(x, x') = \exp(-\gamma\|x - x'\|^2)$ 构造一个有限维的显式特征映射：

$$
\phi(x) = \sqrt{\tfrac{2}{D}}\,\cos(Wx + b),\qquad W_{ij}\sim\mathcal N(0,\ 2\gamma),\quad b_j\sim U(0, 2\pi)
$$

根据 Bochner 定理，$\phi(x)^\top\phi(x') \approx k(x, x')$，而且 $D$ 越大，近似得越准。所以实现起来很简单，先把数据映射成 $\phi(x)$，再交给任务 1.2 的 `train_linear_svm` 训练就行。

这里先用 PCA 把像素降到 100 维，再映射到 $D = 8192$ 维。$\gamma$ 取 $1 / (d\cdot\mathrm{Var}[z])$，和 sklearn 中 `gamma="scale"` 的取法相同。另外，$\|\phi(x)\|\approx 1$，比 3072 维像素特征的范数小得多，所以学习率也要相应调大一些。

In [ ]:
# TODO: 任务1.3 - 随机傅里叶特征
class RandomFourierFeatures:
    """
    RBF 核的随机傅里叶特征映射

    Args:
        d_in: 输入维度
        d_out: 特征维度 D
        gamma: RBF 核参数 γ

    提示：
    1. W 的形状为 [d_in, d_out]，元素 ~ N(0, 2γ)，即 torch.randn(...) * sqrt(2γ)
    2. b 的形状为 [d_out]，元素 ~ U(0, 2π)，即 torch.rand(...) * 2π
    3. __call__ 返回 sqrt(2/D) * cos(X @ W + b)
    """
    def __init__(self, d_in, d_out, gamma, device=device, seed=SEED):
        g = torch.Generator(device=device).manual_seed(seed)
        # ========== 开始编写代码 ==========



        # ========== 结束编写代码 ==========

    def __call__(self, X):
        # ========== 开始编写代码 ==========



        # ========== 结束编写代码 ==========
        pass

# 测试：φ(x)·φ(x') 是否近似 exp(-γ||x - x'||²)
def test_rff():
    print("测试 RandomFourierFeatures")
    print("=" * 50)
    gamma = 0.5
    rff = RandomFourierFeatures(3, 20000, gamma)
    A = torch.randn(5, 3, device=device)
    B = torch.randn(5, 3, device=device)
    approx = (rff(A) * rff(B)).sum(1)
    exact = torch.exp(-gamma * ((A - B) ** 2).sum(1))
    print("近似核值:", [round(v, 3) for v in approx.tolist()])
    print("精确核值:", [round(v, 3) for v in exact.tolist()])
    print(f"最大误差: {(approx - exact).abs().max().item():.4f}（应 < 0.05）")
    print("=" * 50)

test_rff()

In [ ]:
t = time.time()
mean_p, V_p = pca_fit(x_tr[sub], 100)
z_tr, z_te = pca_transform(x_tr[sub], mean_p, V_p), pca_transform(x_te, mean_p, V_p)
gamma = 1.0 / (z_tr.shape[1] * z_tr.var())
rff = RandomFourierFeatures(z_tr.shape[1], 8192, gamma.item())
svm_rff = train_linear_svm(rff(z_tr), y_tr[sub], lam=1e-5, epochs=30, lr=0.3)
record("Part 1", "PCA-100 + RFF-8192 + linear SVM (~ RBF SVM)", "acc", svm_accuracy(svm_rff, rff(z_te), y_te), time.time() - t)

### 任务1.4（选做）：特征工程

换一组特征试试：把图片转成灰度后做二维傅里叶变换，取对数幅度谱作为特征，一共 1024 维（`fft_features` 已经写好）。在这组特征上训练线性 SVM，和像素特征的结果比较一下。绘图代码会画出 REAL 和 FAKE 的平均频谱，可以观察两者有什么区别。

In [ ]:
def fft_features(T):
    gray = T.mean(1)
    gray = gray - gray.mean((1, 2), keepdim=True)
    spec = torch.fft.fftshift(torch.fft.fft2(gray), dim=(1, 2)).abs()
    return torch.log1p(spec).flatten(1)

f_tr, f_te = fft_features(T_tr), fft_features(T_te)
f_mean, f_std = f_tr[sub].mean(0), f_tr[sub].std(0) + 1e-6
f_tr, f_te = (f_tr - f_mean) / f_std, (f_te - f_mean) / f_std

fig, axes = plt.subplots(1, 3, figsize=(11, 3.4))
m_real = f_tr[y_tr == 0].mean(0).view(32, 32).cpu()
m_fake = f_tr[y_tr == 1].mean(0).view(32, 32).cpu()
for ax, img, title, cmap in zip(axes, (m_real, m_fake, m_fake - m_real),
                                ("REAL mean log-spectrum", "FAKE mean log-spectrum", "FAKE - REAL"), ("viridis", "viridis", "coolwarm")):
    im = ax.imshow(img, cmap=cmap)
    ax.set_title(title)
    ax.set_axis_off()
    fig.colorbar(im, ax=ax, fraction=0.046)
savefig("spectrum")

t = time.time()
svm_fft = train_linear_svm(f_tr[sub], y_tr[sub])
record("Part 1", "FFT features + linear SVM", "acc", svm_accuracy(svm_fft, f_te, y_te), time.time() - t)

## Part 2: K-Means 聚类与异常检测

### 任务2.1：实现 K-Means

K-Means 要让每个样本到所属簇中心的平方距离之和尽量小：

$$
\min_{C,\,\mu}\ \sum_{k=1}^{K}\sum_{x_i\in C_k}\|x_i - \mu_k\|^2
$$

Lloyd 算法在两步之间反复交替，直到收敛：先把每个样本分给离它最近的中心，再把每个中心更新为簇内样本的均值。初始中心用 k-means++ 来选，第一个中心随机挑一个样本，之后的每个中心以"到最近已有中心的距离平方"为权重随机抽取。

In [ ]:
# TODO: 任务2.1 - 实现 k-means++ 初始化与 Lloyd 迭代
def kmeans_pp_init(X, K, generator):
    """
    k-means++ 初始化

    Args:
        X: torch.Tensor, shape [N, d]
        K: 簇数
        generator: torch.Generator，用于随机抽样

    Returns:
        centroids: torch.Tensor, shape [K, d]

    提示：
    1. 第一个中心：torch.randint(len(X), (1,), generator=generator, device=X.device)
    2. 之后每个中心：d2 = 到已有中心的最小距离平方（torch.cdist(X, centroids).min(1).values ** 2），
       按 d2 / d2.sum() 的概率用 torch.multinomial 抽取一个样本
    """
    # ========== 开始编写代码 ==========



    # ========== 结束编写代码 ==========
    pass


def kmeans(X, K, iters=100, seed=SEED):
    """
    Lloyd 算法

    Args:
        X: torch.Tensor, shape [N, d]
        K: 簇数
        iters: 最大迭代次数

    Returns:
        centroids: torch.Tensor, shape [K, d]
        assign: torch.LongTensor, shape [N]，每个样本所属的簇

    提示：
    1. 分配：torch.cdist(X, centroids).argmin(1)
    2. 更新：用 torch.zeros(K, d).index_add_(0, assign, X) 累加，再除以 torch.bincount(assign, minlength=K)；
       空簇（计数为 0）保留原中心
    3. 中心不再变化（torch.allclose）则提前结束
    """
    generator = torch.Generator(device=X.device).manual_seed(seed)
    centroids = kmeans_pp_init(X, K, generator)
    # ========== 开始编写代码 ==========



    # ========== 结束编写代码 ==========
    return centroids, assign

# 测试：三个分得很开的高斯簇
def test_kmeans():
    print("测试 kmeans")
    print("=" * 50)
    g = torch.Generator().manual_seed(0)
    centers = torch.tensor([[0.0, 0.0], [8.0, 0.0], [0.0, 8.0]])
    labels = torch.arange(3).repeat_interleave(200)
    pts = (centers[labels] + torch.randn(600, 2, generator=g)).to(device)
    centroids, assign = kmeans(pts, 3)
    purity = sum(torch.bincount(labels.to(device)[assign == k], minlength=3).max().item() for k in range(3)) / 600
    print(f"簇中心:\n{centroids.cpu()}")
    print(f"簇纯度: {purity:.3f}（应 > 0.95）")
    print("=" * 50)

test_kmeans()

### 任务2.2：聚类能分开真图和假图吗

在 PCA-50 空间里把训练子集聚成 $K=2$ 类，然后看簇标签和真假标签有多少是对得上的。簇的编号是随意分配的，所以两种对应方式都算一遍，取较高的那个。

下面会把每个簇的样本画出来，可以看看聚类实际上是按什么标准把图片分开的。

In [ ]:
t = time.time()
mean_50, V_50 = pca_fit(x_tr[sub], 50)
z50_tr, z50_te = pca_transform(x_tr[sub], mean_50, V_50), pca_transform(x_te, mean_50, V_50)
centroids2, _ = kmeans(z50_tr, 2)
pred2 = torch.cdist(z50_te, centroids2).argmin(1)
agreement = max((pred2 == y_te).float().mean().item(), (pred2 != y_te).float().mean().item())
record("Part 2", "2-means on PCA-50, cluster/label agreement", "acc", agreement, time.time() - t)

fig, axes = plt.subplots(2, 10, figsize=(14, 3.2))
for row in range(2):
    idx = torch.nonzero(pred2 == row).squeeze(1)
    idx = idx[torch.randperm(len(idx))[:10]]
    for ax, i in zip(axes[row], idx):
        ax.imshow(X_te[i].permute(1, 2, 0).cpu())
        ax.set_axis_off()
        ax.set_title(f"cluster {row} / {CLASS_NAMES[y_te[i]]}", fontsize=8)
savefig("kmeans_clusters")

fig, ax = plt.subplots(figsize=(5, 4))
for label, color in ((0, "tab:blue"), (1, "tab:red")):
    m = y_te == label
    ax.scatter(z50_te[m, 0].cpu(), z50_te[m, 1].cpu(), s=2, alpha=0.3, c=color, label=CLASS_NAMES[label])
ax.set_title("test set, first two principal components")
ax.legend(markerscale=6)
savefig("pca_scatter")

### 任务2.3：异常检测与 AUROC

现在换个思路：训练时只用真实照片，把假图当作异常来检测。具体来说，在 PCA-100 空间里对真图跑一个 $K=64$ 的 K-Means，测试图片到最近簇中心的距离就是它的异常分数。

异常分数只能用来排序，本身并不直接告诉我们一张图是真是假，所以这里用 AUROC 来评价。它的含义是随机抽一张假图和一张真图，假图分数更高的概率。0.5 相当于随机猜，1 表示排序完全正确。请按下面的公式，用排名来实现 AUROC：

$$
\mathrm{AUROC} = \frac{\sum_{i\in\text{正类}} r_i - n_+(n_++1)/2}{n_+\,n_-}
$$

其中 $r_i$ 是所有分数从小到大排序后的名次（从 1 开始），正类为 FAKE。分数相同的情况可以不考虑。

作为对照，下面也给出了用 PCA 重构误差作为异常分数的结果。

In [ ]:
# TODO: 任务2.3 - 实现 AUROC
def auroc(scores, labels):
    """
    Args:
        scores: torch.Tensor, shape [N]，异常分数，越大越可能是正类
        labels: torch.Tensor, shape [N]，取值 0/1，1 为正类

    Returns:
        float

    提示：
    1. order = scores.argsort()；排名 ranks[order] = 1..N
    2. 代入上面的公式
    """
    # ========== 开始编写代码 ==========



    # ========== 结束编写代码 ==========
    pass

def test_auroc():
    print("测试 auroc")
    print("=" * 50)
    s = torch.tensor([0.1, 0.4, 0.35, 0.8])
    l = torch.tensor([0, 0, 1, 1])
    print(f"你的输出: {auroc(s, l):.4f}  期望: 0.7500")
    print(f"完美排序: {auroc(torch.tensor([0.1, 0.2, 0.9, 0.95]), l):.4f}  期望: 1.0000")
    print(f"完全反序: {auroc(torch.tensor([0.9, 0.95, 0.1, 0.2]), l):.4f}  期望: 0.0000")
    print("=" * 50)

test_auroc()

In [ ]:
real_idx = fit_idx[y_tr[fit_idx] == 0]
t = time.time()
mean_r, V_r = pca_fit(x_tr[real_idx], 100)
zr_tr, zr_te = pca_transform(x_tr[real_idx], mean_r, V_r), pca_transform(x_te, mean_r, V_r)
centroids64, _ = kmeans(zr_tr, 64)
score_km = torch.cdist(zr_te, centroids64).min(1).values
record("Part 2", "K-Means(64) centroid distance, REAL only", "auroc", auroc(score_km, y_te), time.time() - t)

t = time.time()
score_pca = ((x_te - pca_inverse(zr_te, mean_r, V_r)) ** 2).mean(1)
record("Part 2", "PCA-100 reconstruction error, REAL only", "auroc", auroc(score_pca, y_te), time.time() - t)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.2))
for ax, (score, name) in zip(axes, ((score_km, "K-Means distance"), (score_pca, "PCA reconstruction error"))):
    for label in (0, 1):
        ax.hist(score[y_te == label].cpu().tolist(), 60, alpha=0.6, density=True, label=CLASS_NAMES[label])
    ax.set_title(f"{name} as anomaly score")
    ax.legend()
savefig("anomaly_scores")

## Part 3: 前馈神经网络

### 辅助函数：小批量迭代与评估（已给出）

从这里开始，所有模型都按同一套流程训练：用交叉熵损失做小批量训练，每个 epoch 结束后在验证集上测一次准确率，最终保留验证准确率最高那一轮的参数，然后只在测试集上评估一次。

In [ ]:
def iterate_minibatches(index, batch_size, shuffle=True):
    if shuffle:
        index = index[torch.randperm(len(index), device=index.device)]
    for i in range(0, len(index), batch_size):
        yield index[i:i + batch_size]

@torch.no_grad()
def evaluate(model, X, y, batch_size=2048):
    model.eval()
    correct = 0
    for idx in iterate_minibatches(torch.arange(len(X), device=X.device), batch_size, shuffle=False):
        correct += (model(X[idx]).argmax(1) == y[idx]).sum().item()
    return correct / len(X)

def count_params(model):
    return sum(p.numel() for p in model.parameters())

### 任务3.1：MLP

输入是展平后的 3072 维向量，中间叠几层 `Linear + ReLU`，最后一层输出 2 个 logits。输出端不用再加 softmax，`F.cross_entropy` 内部已经包含了这一步。

In [ ]:
# TODO: 任务3.1 - 实现 MLP
class MLP(nn.Module):
    """
    多层感知机

    Args:
        hidden: 各隐藏层宽度，如 (512, 256)
        num_classes: 输出类别数

    提示：
    1. 先 nn.Flatten()，再依次 nn.Linear(in, h) + nn.ReLU()，最后 nn.Linear(h_last, num_classes)
    2. 用 nn.Sequential 组织各层
    """
    def __init__(self, hidden=(512, 256), num_classes=2):
        super().__init__()
        # ========== 开始编写代码 ==========



        # ========== 结束编写代码 ==========

    def forward(self, x):
        # ========== 开始编写代码 ==========



        # ========== 结束编写代码 ==========
        pass

# 测试
model = MLP((512, 256)).to(device)
out = model(T_tr[:4])
print(f"输出 shape: {tuple(out.shape)}（应为 (4, 2)）")
print(f"参数量: {count_params(model):,}（应为 1,705,218）")

### 任务3.2：训练函数

把完整的训练循环写出来。学习率调度用 `OneCycleLR`，它会让学习率先升后降，创建时需要告诉它总共训练多少步。

In [ ]:
# TODO: 任务3.2 - 实现训练函数
def train_model(model, X, y, X_val, y_val, epochs=EPOCHS, lr=1e-3, optimizer="adamw", batch_size=BATCH, augment=None, log=True):
    """
    Args:
        model: nn.Module
        X, y: 训练数据与标签
        X_val, y_val: 验证数据与标签
        optimizer: "adamw" 或 "sgd"
        augment: 对每个 batch 的输入做数据增强的函数，None 表示不增强

    Returns:
        history: dict，包含每个 epoch 的 "train_loss" 与 "val_acc"（list）
        训练结束后 model 恢复为验证准确率最高一轮的参数

    提示：
    1. adamw: torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=5e-4)
       sgd:   torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    2. 调度器: torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * ceil(N / batch_size))
    3. 每个 batch: 前向 → F.cross_entropy → zero_grad → backward → opt.step() → scheduler.step()
    4. 用 copy.deepcopy(model.state_dict()) 保存最优参数
    """
    model = model.to(device)
    # ========== 开始编写代码 ==========



    # ========== 结束编写代码 ==========
    history = {"train_loss": [], "val_acc": []}
    best_acc, best_state = -1.0, None
    index = torch.arange(len(X), device=X.device)
    for epoch in range(epochs):
        model.train()
        total = 0.0
        for idx in iterate_minibatches(index, batch_size):
            xb, yb = X[idx], y[idx]
            if augment is not None:
                xb = augment(xb)
            # ========== 开始编写代码 ==========



            # ========== 结束编写代码 ==========
            total += loss.item() * len(idx)
        val_acc = evaluate(model, X_val, y_val)
        history["train_loss"].append(total / len(X))
        history["val_acc"].append(val_acc)
        if val_acc > best_acc:
            best_acc, best_state = val_acc, copy.deepcopy(model.state_dict())
        if log:
            print(f"epoch {epoch + 1:2d}  train loss {history['train_loss'][-1]:.4f}  val acc {val_acc:.4f}")
    model.load_state_dict(best_state)
    return history

def run_experiment(part, name, model, epochs=EPOCHS, log=True, **kw):
    """训练 → 在验证集上选最优 epoch → 测试集评价一次 → 记录"""
    t = time.time()
    history = train_model(model, T_tr[fit_idx], y_tr[fit_idx], T_tr[val_idx], y_tr[val_idx], epochs=epochs, log=log, **kw)
    record(part, name, "acc", evaluate(model, T_te, y_te), time.time() - t)
    return history

def plot_histories(histories, name):
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
    for label, h in histories.items():
        axes[0].plot(range(1, len(h["train_loss"]) + 1), h["train_loss"], marker="o", ms=3, label=label)
        axes[1].plot(range(1, len(h["val_acc"]) + 1), h["val_acc"], marker="o", ms=3, label=label)
    axes[0].set_title("training loss"); axes[1].set_title("validation accuracy")
    for ax in axes:
        ax.set_xlabel("epoch"); ax.grid(True); ax.legend(fontsize=8)
    savefig(name)

# 快速测试：少量数据跑 2 个 epoch，损失应下降
small = fit_idx[:2048]
h = train_model(MLP((128,)), T_tr[small], y_tr[small], T_tr[val_idx], y_tr[val_idx], epochs=2, log=True)
print(f"损失下降: {h['train_loss'][-1] < h['train_loss'][0]}")

### 任务3.3：训练 MLP，比较网络规模

下面会训练三个不同规模的 MLP，可以对比一下它们的训练损失和验证准确率有什么不同。

In [ ]:
histories = {}
for hidden in ((256,), (512, 256), (1024, 512, 256)):
    name = f"MLP {hidden}"
    print(f"\n{name}，参数量 {count_params(MLP(hidden)):,}")
    histories[name] = run_experiment("Part 3", name, MLP(hidden))
h_mlp = histories["MLP (512, 256)"]
plot_histories(histories, "mlp_curves")

## Part 4: 卷积神经网络

### 任务4.1：CNN

网络由三个卷积块组成，每块是 `Conv2d(3×3, padding=1) → [BatchNorm2d] → ReLU → MaxPool2d(2)`，通道数依次为 32、64、128。32×32 的输入经过三次池化后变成 4×4，展平后接上分类头 `Linear(128·16, 256) → ReLU → Linear(256, 2)`，`dropout` 加在分类头两个 `Linear` 的前面。当 `use_bn=False, dropout=0` 时，它就是一个最朴素的 CNN。

In [ ]:
# TODO: 任务4.1 - 实现 CNN
class CNN(nn.Module):
    """
    Args:
        channels: 三个卷积块的输出通道数
        use_bn: 是否在每个卷积后加 BatchNorm2d
        dropout: 分类头的 dropout 概率

    提示：
    1. 每个卷积块：nn.Conv2d(c_in, c_out, 3, padding=1), (nn.BatchNorm2d(c_out)), nn.ReLU(), nn.MaxPool2d(2)
    2. 分类头：nn.Flatten(), nn.Dropout(p), nn.Linear(c_last * 4 * 4, 256), nn.ReLU(), nn.Dropout(p), nn.Linear(256, num_classes)
    """
    def __init__(self, channels=(32, 64, 128), use_bn=False, dropout=0.0, num_classes=2):
        super().__init__()
        # ========== 开始编写代码 ==========



        # ========== 结束编写代码 ==========

    def forward(self, x):
        # ========== 开始编写代码 ==========



        # ========== 结束编写代码 ==========
        pass

# 测试
model = CNN().to(device)
out = model(T_tr[:4])
print(f"输出 shape: {tuple(out.shape)}（应为 (4, 2)）")
print(f"参数量: {count_params(model):,}（应为 618,306）；MLP(512, 256) 为 {count_params(MLP((512, 256))):,}")

### 任务4.2：数据增强

对每个 batch 做随机裁剪和随机水平翻转，这是 CIFAR 这类任务里最常用的数据增强。随机裁剪的做法是先在图片四周各补 `pad` 个像素，再随机裁回 32×32。用张量操作实现即可，对 batch 里的样本逐个循环也没问题，只是向量化的写法会快一些。

In [ ]:
# TODO: 任务4.2 - 实现随机裁剪 + 水平翻转
def random_crop_flip(x, pad=4):
    """
    Args:
        x: torch.Tensor, shape [B, 3, 32, 32]
    Returns:
        torch.Tensor, shape [B, 3, 32, 32]

    提示：
    1. F.pad(x, (pad, pad, pad, pad), mode="reflect") 得到 [B, 3, 32+2pad, 32+2pad]
    2. 每个样本随机一个左上角 (dy, dx) ∈ [0, 2pad]，裁出 32×32
    3. 以 0.5 的概率对样本做 .flip(-1)
    """
    # ========== 开始编写代码 ==========



    # ========== 结束编写代码 ==========
    pass

# 测试
xb = T_tr[:8]
aug = random_crop_flip(xb)
print(f"输出 shape: {tuple(aug.shape)}（应与输入一致 {tuple(xb.shape)}）")
same_or_flipped = all(torch.equal(a, b) or torch.equal(a, b.flip(-1)) for a, b in zip(random_crop_flip(xb, pad=0), xb))
print(f"pad=0 时输出等于原图或其水平翻转: {same_or_flipped}")

fig, axes = plt.subplots(2, 8, figsize=(12, 3.2))
for col in range(8):
    for row, img in enumerate((xb[col], aug[col])):
        axes[row, col].imshow(((img * STD[0] + MEAN[0]).clamp(0, 1)).permute(1, 2, 0).cpu())
        axes[row, col].set_axis_off()
axes[0, 0].set_title("original", fontsize=8); axes[1, 0].set_title("augmented", fontsize=8)
savefig("augmentation")

### 任务4.3：优化与正则化

用同一个 CNN 结构做三组对比实验：

1. 基线：不加 BatchNorm、dropout 和数据增强，优化器用 AdamW。
2. 换优化器：其他设置和基线相同，改用 SGD + momentum，学习率 0.05。
3. 加正则化：加上 BatchNorm、0.3 的 dropout 和数据增强，优化器用 AdamW。

In [ ]:
cnn_histories = {}
cnn_histories["CNN baseline (AdamW)"] = run_experiment("Part 4", "CNN baseline (AdamW)", CNN())
cnn_histories["CNN (SGD+momentum)"] = run_experiment("Part 4", "CNN (SGD + momentum, lr=0.05)", CNN(), optimizer="sgd", lr=0.05)
cnn_histories["CNN + BN + dropout + aug"] = run_experiment("Part 4", "CNN + BatchNorm + dropout + augmentation", CNN(use_bn=True, dropout=0.3), augment=random_crop_flip)
plot_histories({"MLP (512, 256)": h_mlp, **cnn_histories}, "cnn_curves")

## Part 5: 结果汇总与分析

### 汇总表与阶梯图

In [ ]:
print(f"{'part':8s} {'method':48s} {'metric':6s} {'value':>8s} {'sec':>8s}")
print("-" * 84)
for r in results:
    print(f"{r['part']:8s} {r['method']:48s} {r['metric']:6s} {r['value']:8.4f} {r['seconds']:8.1f}")

with open(OUT_DIR / "results.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(results[0].keys()))
    writer.writeheader()
    writer.writerows(results)
print(f"\n已保存到 {OUT_DIR / 'results.csv'}")

colors = {"Part 1": "tab:blue", "Part 2": "tab:orange", "Part 3": "tab:green", "Part 4": "tab:red"}
fig, ax = plt.subplots(figsize=(9, 0.42 * len(results) + 1))
ax.barh([f"{r['method']}  [{r['metric']}]" for r in results], [r["value"] for r in results], color=[colors[r["part"]] for r in results])
ax.axvline(0.5, color="k", lw=0.8, ls="--")
ax.set_xlim(0.4, 1.0)
ax.invert_yaxis()
for i, r in enumerate(results):
    ax.text(r["value"] + 0.004, i, f"{r['value']:.3f}", va="center", fontsize=8)
ax.set_title("CIFAKE real vs. fake: test accuracy (AUROC for anomaly detectors)", fontsize=10)
savefig("ladder")

### 问题分析

请结合自己的实验结果回答下面的问题，每题写三到五句话即可。

**问题 1.** 在原始像素上，线性 SVM 的准确率并不高，用 RFF 近似 RBF 核之后有明显提升，这是为什么？RFF 的维度 $D$ 和参数 $\gamma$ 分别会怎样影响结果？可以补充实验来说明。

**回答：**


**问题 2.** $K=2$ 的 K-Means 没能把真图和假图分开。结合任务 2.2 中各簇的样本和 PCA 散点图，说说聚类实际上是按什么把图片分开的，以及这反映出无监督学习有什么局限。

**回答：**


**问题 3.** 只用真图训练的异常检测器，AUROC 比有监督分类器低很多，但仍然高于 0.5，这说明了什么？为什么 PCA 重构误差比 K-Means 中心距离更好用？

**回答：**


**问题 4.** `CNN()` 的参数量比 `MLP((512, 256))` 还少，准确率却高出一大截。请从归纳偏置（局部性、权值共享、平移等变性）的角度解释原因。

**回答：**


**问题 5.** 对比 Part 4 的三组实验，换优化器和加正则化分别让训练损失曲线和验证准确率曲线发生了怎样的变化？哪一组的训练损失和验证准确率之间差距最小，为什么？

**回答：**